# Chapter-9: Retrievers

# <font color="#800000">Source Based LangChain Retrievers</font>

## <font color="#3366FF">Vector Store As Retrievers</font>

### Vector DB Setup For Example

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## List of LangChain Documents
documents = [
    Document(
        page_content="Mars is often called the Red Planet due to its reddish appearance.",
        metadata={"topic": "space"}
    ),
    Document(
        page_content="The Hubble Space Telescope has provided stunning images of distant galaxies.",
        metadata={"topic": "space"}
    ),
    Document(
        page_content="Saturn is famous for its large and bright ring system.",
        metadata={"topic": "space"}
    ),
    Document(
        page_content="Regular exercise helps improve cardiovascular health and stamina.",
        metadata={"topic": "health"}
    ),
    Document(
        page_content="A balanced diet rich in fruits and vegetables boosts the immune system.",
        metadata={"topic": "health"}
    ),
    Document(
        page_content="Sleep is essential for maintaining both physical and mental well-being.",
        metadata={"topic": "health"}
    )
]

## Create Chroma Vector Store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=OpenAIEmbeddings()
)


/Users/ankur/Desktop/Langchain_Review/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Convert Vector Store To Retriever

In [2]:
## Convert vector store into retriever
retriever = vectorstore.as_retriever()


### Invoke Retriever

In [3]:
## Invoke Retrievers
retriever.invoke("What is importance of fruits")


[Document(id='70461b18-f62f-4e8b-ae23-8b36d5199e99', metadata={'topic': 'health'}, page_content='A balanced diet rich in fruits and vegetables boosts the immune system.'),
 Document(id='08108269-2446-4554-aa65-1a2353d64209', metadata={'topic': 'health'}, page_content='Regular exercise helps improve cardiovascular health and stamina.'),
 Document(id='9888c6af-6d55-496e-8af5-b3fd617f46a4', metadata={'topic': 'health'}, page_content='Sleep is essential for maintaining both physical and mental well-being.'),
 Document(id='178d12f9-70b3-45b2-b550-b17057a9ecdc', metadata={'topic': 'space'}, page_content='Saturn is famous for its large and bright ring system.')]

### Configuring Vector Store Retriever

#### Search Type As “similarity”

In [4]:
## Convert vector store into retriever
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

## Invoke Retrievers
retriever.invoke("What is importance of fruits")


[Document(id='70461b18-f62f-4e8b-ae23-8b36d5199e99', metadata={'topic': 'health'}, page_content='A balanced diet rich in fruits and vegetables boosts the immune system.'),
 Document(id='08108269-2446-4554-aa65-1a2353d64209', metadata={'topic': 'health'}, page_content='Regular exercise helps improve cardiovascular health and stamina.')]

#### Search Type As “similarity_score_threshold”

In [5]:
## Convert vector store into retriever
retriever = vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.7})

## Invoke Retrievers
retriever.invoke("What is importance of fruits")


[Document(id='70461b18-f62f-4e8b-ae23-8b36d5199e99', metadata={'topic': 'health'}, page_content='A balanced diet rich in fruits and vegetables boosts the immune system.')]

#### Search Type As “mmr”

In [6]:
## Convert vector store into retriever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3})

## Invoke Retrievers
retriever.invoke("What is importance of fruits")


[Document(id='70461b18-f62f-4e8b-ae23-8b36d5199e99', metadata={'topic': 'health'}, page_content='A balanced diet rich in fruits and vegetables boosts the immune system.'),
 Document(id='9888c6af-6d55-496e-8af5-b3fd617f46a4', metadata={'topic': 'health'}, page_content='Sleep is essential for maintaining both physical and mental well-being.'),
 Document(id='ff2001c4-b219-4775-9a27-85b3cfee2cbe', metadata={'topic': 'space'}, page_content='Mars is often called the Red Planet due to its reddish appearance.')]

### Use Of Retriever In RAG

In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Create Prompt Template
prompt = ChatPromptTemplate.from_template(
    """Answer following question using only below context:

Context: {context}

Question: {question}"""
)

## Function To Concatenate Documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

## Input question
question = "What is importance of fruits?"

## Convert vector store into retriever
retrieved_docs = retriever.invoke(question)

## Format documents into context string
context = format_docs(retrieved_docs)

## Create prompt input
prompt_input = {
    "context": context,
    "question": question
}

## Format prompt
formatted_prompt = prompt.invoke(prompt_input)

## Invoke LLM with formatted prompt
llm_response = llm.invoke(formatted_prompt)

## Print result
print(llm_response.content)


Fruits are important because, as part of a balanced diet rich in fruits and vegetables, they help boost the immune system.


## <font color="#3366FF">Tavily Search API Retriever (Internet Search)</font>

### Example Of Tavily Search Retriever

In [8]:
import os
from dotenv import load_dotenv
from langchain_classic.retrievers import TavilySearchAPIRetriever

## Load Tavily API Key into Environment Variable
load_dotenv()
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")

## Initialize Tavily Search Retriever
retriever = TavilySearchAPIRetriever(k=3)

## Invoke Retriever
retriever.invoke("How are Apple stocks performing today?")


[Document(metadata={'title': 'Buy or Sell Apple Stock - AAPL Stock Price Quote & News | Robinhood', 'source': 'https://robinhood.com/us/en/stocks/AAPL', 'score': 0.7306941, 'id': 'e072a2-00', 'images': []}, page_content="## Stock Snapshot\n\nAs of today, Apple(AAPL) shares are valued at $336.08. The company's market cap stands at 4.91T, with a P/E ratio of 38.59 and a dividend yield of 0.31%.\n\nAs of 2026-10-11, Apple(AAPL) stock has fluctuated between $330.70 and $338.61. The current price stands at $336.08, placing the stock +1.6% above today's low and -0.7% off the high.\n\nThe Apple(AAPL)'s current trading volume is 37.88M, compared to an average daily volume of 36.31M. [...] In the last year, Apple(AAPL) shares hit a 52-week high of $345.34 and a 52-week low of $243.42.\n\nIn the last year, Apple(AAPL) shares hit a 52-week high of $345.34 and a 52-week low of $243.42.\n\n## AAPL News\n\nApple revealed in a regulatory filing that it has reached an agreement to bring on team member

### Use Of Tavily Search Retriever In RAG Chain

In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import TavilySearchAPIRetriever

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Create Prompt Template
prompt = ChatPromptTemplate.from_template(
    """Answer following question using only below context:

Context: {context}

Question: {question}"""
)

## Function To Concatenate Documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Initialize Tavily Search Retriever
retriever = TavilySearchAPIRetriever(k=3)

## Input question
question = "How are Apple stocks performing today?"

## Invoke retriever
retrieved_docs = retriever.invoke(question)

## Format documents into context string
context = format_docs(retrieved_docs)

## Create prompt input
prompt_input = {
    "context": context,
    "question": question
}

## Format prompt
formatted_prompt = prompt.invoke(prompt_input)

## Invoke LLM with formatted prompt
llm_response = llm.invoke(formatted_prompt)

## Print result
print(llm_response.content)


Apple is trading around $336.08 today. Key intraday stats:
- Day's range: $330.70 — $338.61 (current price is ~1.6% above the day's low and ~0.7% below the day's high)
- Current volume: 37.88M vs. average daily volume ~36.31M
- Market cap: ~$4.91T; P/E ~38.6

In short: trading modestly off its intraday high with volume slightly above average.


## <font color="#3366FF">Wikipedia Retriever</font>

### Example Of Wikipedia Retriever

In [10]:
from langchain_classic.retrievers import WikipediaRetriever

## Initialize Wikipedia Retriever
retriever = WikipediaRetriever(top_k_results=2)

## Invoke Retriever
retriever.invoke("What is the population of France?")


[Document(metadata={'title': 'Demographics of France', 'summary': 'The demography of France is monitored by the Institut national d\'études démographiques (INED) and the Institut national de la statistique et des études économiques (INSEE). As of 1 January 2026,  in Metropolitan France 66,792,845 people lived, while 2,289,151 lived in overseas France, for a total of 69,081,996 inhabitants in the French Republic. In the 2010s and until 2017, the population of France grew by 1 million people every three years – an average annual increase of 340,000 people, or +0.6%.\nFrance was historically Europe\'s most populous country. During the Middle Ages, more than one-quarter of Europe\'s total population was French; by the seventeenth century, this had decreased slightly to one-fifth. By the beginning of the nineteenth century, other European countries, such as Germany and Russia, had caught up with France and overtaken it in number of people. The country\'s population sharply increased with th

# <font color="#800000">Retrieval Strategy Based Retrievers</font>

## <font color="#3366FF">MultiQuery Retriever</font>

### Example Of MultiQuery Retriever

In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_classic.retrievers import TavilySearchAPIRetriever
import logging

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Initialize Tavily Search Retriever
retriever=TavilySearchAPIRetriever(k=2)

## Initialize MultiQuery Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)
## Enable Logging
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

## Invoke MultiQuery Retriever
docs = multi_query_retriever.invoke("Is this the right time to purchase Apple stocks?")

## Print Retrieved Documents
for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


INFO:langchain_classic.retrievers.multi_query:Generated queries: ["Is now a good time to buy Apple (AAPL) stock given the company's recent earnings, guidance, and current market conditions?", "What are the risks, potential returns, and recommended entry strategies for purchasing Apple shares at today's price?", "How does Apple's current valuation compare to its historical multiples and peers, and does that indicate an attractive entry point for long-term investors?"]


Document 1
Content: StockInvest.us
StockInvest.us
StockInvest.us
StockInvest.us
Profile

StockInvest.us
StockInvest.us

# Apple Stock Forecast

etoro logo

## Should you buy AAPL stock?

Last updated:
Sep 25, 2026, 22:18 UTC

Based on market data through
Sep 25, 2026,
StockInvest.us rates Apple (AAPL) as
a Buy Candidate with a technical score of
3.96 on its -10 to +10 scale.
This is an upgrade from Hold in the previous daily evaluation.

Use StockInvest.us often?

AI Analysis [...] The Apple stock price gained 1.53% on the last trading day (Friday, 25th Sep 2026), rising from $335.92 to $341.07. During the last trading day the stock fluctuated 2.13% from a day low at $334.53 to a day high of $341.67. The price has risen in 6 of the last 10 days and is up by 2.65% over the past 2 weeks. Volume has increased on the last day along with the price, which is a positive technical sign, and, in total, 5 million more shares were traded than the day before. In total, 29 million
---
Document 2
Co

### Use Of MultiQuery Retriever In RAG

In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import TavilySearchAPIRetriever
from langchain_classic.retrievers import MultiQueryRetriever

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Create Prompt Template
prompt = ChatPromptTemplate.from_template(
    """Answer following question using only below context:

Context: {context}

Question: {question}"""
)

## Function To Concatenate Documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Input question
question = "How are Apple stocks performing today?"

## Initialize MultiQuery Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=TavilySearchAPIRetriever(k=3), llm=llm
)

## Invoke MultiQuery Retriever
retrieved_docs = multi_query_retriever.invoke(question)

## Format documents into context string
context = format_docs(retrieved_docs)

## Create prompt input
prompt_input = {
    "context": context,
    "question": question
}

## Format prompt
formatted_prompt = prompt.invoke(prompt_input)

## Invoke LLM with formatted prompt
llm_response = llm.invoke(formatted_prompt)

## Print result
print(llm_response.content)


INFO:langchain_classic.retrievers.multi_query:Generated queries: ["What's AAPL's current stock price, today's percent change, and intraday high/low?", "How did Apple Inc. (AAPL) perform in today's trading session, including trading volume and after-hours movement?", "What news, analyst updates, or market sentiment are driving Apple's stock performance today?"]


Quick summary (from the provided data):

- Current price: $336.08.
- Change vs. previous close ($340.42): down about 1.3%.
- Day's range: $330.70 – $338.61 (current price is ~+1.6% above the day's low and ~‑0.7% below the day's high).
- Open: $331.79 (so ~+1.3% vs. the open).
- Volume: 37.88M vs. average ~36.31M (slightly above average).
- After-hours: roughly $336.05 (little changed).
- Context metrics: market cap ~$4.91T, P/E ~38.6, dividend yield ~0.31%. 52-week range: $243.42–$345.34 (current ≈4.8% below the 52‑week high).

Bottom line: trading slightly down versus yesterday’s close but off the intraday low and with volume a bit above normal.


## <font color="#3366FF">Contextual Compression Retriever</font>

### Vector Store Retriever Setup For Examples

In [13]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## List of LangChain Documents
documents = [
    Document(
        page_content="""Top Analyst Stocks Top Smart Score Stocks Dividend CenterBest Dividend Stocks Top Insiders Stocks Top Stocks Compare stocks Stock Screener Penny Stock Screener Top Analyst Stocks AI Analyst Top Stocks Top Insiders Stocks Compare Stocks Trending Stocks Dividend Stocks Dividend Calculator Best Dividend Stocks Best High Yield Dividend Stocks Dividend Stock Comparison Dividend Calculator STOCKS The pressures on Apple’s stock price this year have come from a litany of sources, ranging from worries over tariffs that could inflate iPhone manufacturing costs to frustration with the company’s slow progress in AI. Overall we can expect increase of 1-3% 
over next 1 month after launch of Iphone 17.""",
    ),
    Document(
        page_content="""The **Apple** stock price increased **0.95%** on the final trading day (Tuesday, 26th Aug 2025), rising from **$227.16** to **$229.31**. The **Apple** stock holds buy signals from both short and long-term Moving Averages giving a positive forecast for the stock. Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term. Should I buy or sell Apple Stock? Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term.""",
    ),
    Document(
        page_content="""Recent price changes and earnings estimate revisions indicate this would be a good stock for momentum investors with a Momentum Score of A.""",
    )
]

## Create Chroma Vector Store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=OpenAIEmbeddings()
)

## Convert vector store into retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## Invoke Retriever
docs=retriever.invoke("How much Apple stocks increased?")

## Print document chunks one by one
for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: The **Apple** stock price increased **0.95%** on the final trading day (Tuesday, 26th Aug 2025), rising from **$227.16** to **$229.31**. The **Apple** stock holds buy signals from both short and long-term Moving Averages giving a positive forecast for the stock. Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term. Should I buy or sell Apple Stock? Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term.
---
Document 2
Content: Top Analyst Stocks Top Smart Score Stocks Dividend CenterBest Dividend Stocks Top Insiders Stocks Top Stocks Compare stocks Stock Screener Penny Stock Screener Top Analyst Stocks AI Analyst Top Stocks Top Inside

### Example Of LLMChainExtractor

In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Initialize LLM Chain Extractor with LLM
compressor = LLMChainExtractor.from_llm(llm)

## Initialize Contextual Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

## Invoke Contextual Compression Retriever
compressed_docs = compression_retriever.invoke("How much Apple stocks increased?")

## Print retrieved documents one by one
for i, doc in enumerate(compressed_docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: The **Apple** stock price increased **0.95%** on the final trading day (Tuesday, 26th Aug 2025), rising from **$227.16** to **$229.31**.
---
Document 2
Content: Overall we can expect increase of 1-3% 
over next 1 month after launch of Iphone 17.
---


### Example Of LLMChainFilter

In [15]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainFilter

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Initialize LLM Chain Filter with LLM
compressor = LLMChainFilter.from_llm(llm)

## Initialize Contextual Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

## Invoke Contextual Compression Retriever
compressed_docs = compression_retriever.invoke(
    "How much Apple stocks increased?"
)

## Print retrieved documents one by one
for i, doc in enumerate(compressed_docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: The **Apple** stock price increased **0.95%** on the final trading day (Tuesday, 26th Aug 2025), rising from **$227.16** to **$229.31**. The **Apple** stock holds buy signals from both short and long-term Moving Averages giving a positive forecast for the stock. Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term. Should I buy or sell Apple Stock? Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term.
---
Document 2
Content: Top Analyst Stocks Top Smart Score Stocks Dividend CenterBest Dividend Stocks Top Insiders Stocks Top Stocks Compare stocks Stock Screener Penny Stock Screener Top Analyst Stocks AI Analyst Top Stocks Top Inside

### Example Of LLMListwiseRerank

In [16]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Creating OpenAI Chat Model instance
llm = ChatOpenAI(model="gpt-5-mini", api_key=os.getenv('OPENAI_API_KEY'))

## Initialize LLM Listwise Rerank with LLM
compressor = LLMListwiseRerank.from_llm(llm, top_n=2)

## Initialize Contextual Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

## Invoke Contextual Compression Retriever
compressed_docs = compression_retriever.invoke("How much Apple stocks increased?")

## Print retrieved documents one by one
for i, doc in enumerate(compressed_docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: The **Apple** stock price increased **0.95%** on the final trading day (Tuesday, 26th Aug 2025), rising from **$227.16** to **$229.31**. The **Apple** stock holds buy signals from both short and long-term Moving Averages giving a positive forecast for the stock. Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term. Should I buy or sell Apple Stock? Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term.
---
Document 2
Content: Top Analyst Stocks Top Smart Score Stocks Dividend CenterBest Dividend Stocks Top Insiders Stocks Top Stocks Compare stocks Stock Screener Penny Stock Screener Top Analyst Stocks AI Analyst Top Stocks Top Inside

### Example Of EmbeddingsFilter

In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initializing OpenAI Embedding
embeddings = OpenAIEmbeddings()

## Initialize Embedding Filter
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.85, k=2)

## Initialize Contextual Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever
)

## Invoke Contextual Compression Retriever
compressed_docs = compression_retriever.invoke("How much Apple stocks increased?")

## Print retrieved documents one by one
for i, doc in enumerate(compressed_docs):
    print(f"Document{i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document1
Content: The **Apple** stock price increased **0.95%** on the final trading day (Tuesday, 26th Aug 2025), rising from **$227.16** to **$229.31**. The **Apple** stock holds buy signals from both short and long-term Moving Averages giving a positive forecast for the stock. Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term. Should I buy or sell Apple Stock? Several short-term signals, along with a general good trend, are positive and we conclude that the current level may hold a buying opportunity as there is a fair chance for **Apple** stock to perform well in the short-term.
---


## <font color="#3366FF">Parent Document Retriever</font>

### Example Of ParentDocumentRetriever

#### Setup Of ParentDocumentRetriever

In [18]:
import os
from dotenv import load_dotenv
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import LocalFileStore
from langchain_classic.storage._lc_store import create_kv_docstore


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## List of LangChain Documents
docs = [
    Document(
        page_content="""A black hole is a region in space where gravity is so strong that nothing, not even light, can escape from it. Black holes form when very massive stars collapse at the end of their life cycles. They have an invisible boundary called the event horizon, beyond which anything that crosses is trapped forever. Although black holes cannot be seen directly, scientists detect them by observing their effects on nearby stars and gas. They are among the most mysterious objects in the universe, challenging our understanding of space, time, and physics."""
    ),
    Document(
        page_content="""Global warming refers to the long-term rise in Earth’s average temperature due to the buildup of greenhouse gases in the atmosphere. Human activities such as burning coal, oil, and gas, along with deforestation, release large amounts of carbon dioxide and other heat-trapping gases. This causes the planet to heat up beyond natural levels, leading to melting glaciers, rising sea levels, more frequent heatwaves, and changes in rainfall patterns. If left unchecked, global warming threatens ecosystems, human health, and the balance of life on Earth."""
    )
]

## Initialize LocalFileStore with 
file_store = LocalFileStore("./document_store")

## Initialize Key Value Doc Store
parent_doc_store = create_kv_docstore(file_store)

## Initialize Chroma Store
vectorstore = Chroma(embedding_function=OpenAIEmbeddings(), persist_directory="./vector_store")
## Initialize a RecursiveCharacterTextSplitter for Child Splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap = 50)

## Initialize Parent Document Retriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=parent_doc_store,
    child_splitter=child_splitter
)

## Add Documents to Parent Document Retriever
parent_retriever.add_documents(docs, ids=None)


#### Query Vector Store

In [19]:
## Perform Similarity Search on Base Vector Store
chunks = vectorstore.similarity_search("What is impact of global warming?")

## Print retrieved chunks one by one
for i, doc in enumerate(chunks):
    print(f"Chunk {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Chunk 1
Content: global warming threatens ecosystems, human health, and the balance of life on Earth.
---
Chunk 2
Content: and changes in rainfall patterns. If left unchecked, global warming threatens ecosystems, human
---
Chunk 3
Content: to heat up beyond natural levels, leading to melting glaciers, rising sea levels, more frequent
---
Chunk 4
Content: and other heat-trapping gases. This causes the planet to heat up beyond natural levels, leading to
---


#### Invoke ParentDocumentRetriever

In [20]:
## Invoke Parent Document Retriever
parent_docs = parent_retriever.invoke("What is impact of global warming?")

## Print retrieved documents one by one
for i, doc in enumerate(parent_docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: Global warming refers to the long-term rise in Earth’s average temperature due to the buildup of greenhouse gases in the atmosphere. Human activities such as burning coal, oil, and gas, along with deforestation, release large amounts of carbon dioxide and other heat-trapping gases. This causes the planet to heat up beyond natural levels, leading to melting glaciers, rising sea levels, more frequent heatwaves, and changes in rainfall patterns. If left unchecked, global warming threatens ecosystems, human health, and the balance of life on Earth.
---


### Using Parent Splitter

#### Setup Of ParentDocumentRetriever With Parent Splitter

In [21]:
import os
from dotenv import load_dotenv
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import LocalFileStore
from langchain_classic.storage._lc_store import create_kv_docstore


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## List of LangChain Documents
docs = [
    Document(
        page_content="""A black hole is a region in space where gravity is so strong that nothing, not even light, can escape from it. Black holes form when very massive stars collapse at the end of their life cycles. They have an invisible boundary called the event horizon, beyond which anything that crosses is trapped forever. Although black holes cannot be seen directly, scientists detect them by observing their effects on nearby stars and gas. They are among the most mysterious objects in the universe, challenging our understanding of space, time, and physics.
        """
    ),
    Document(
        page_content="""Global warming refers to the long-term rise in Earth’s average temperature due to the buildup of greenhouse gases in the atmosphere. Human activities such as burning coal, oil, and gas, along with deforestation, release large amounts of carbon dioxide and other heat-trapping gases. This causes the planet to heat up beyond natural levels, leading to melting glaciers, rising sea levels, more frequent heatwaves, and changes in rainfall patterns. If left unchecked, global warming threatens ecosystems, human health, and the balance of life on Earth.
        """
    )
]

## Initialize LocalFileStore with 
file_store = LocalFileStore("./document_store")

## Initialize Key Value Doc Store
parent_doc_store = create_kv_docstore(file_store)

## Initialize Chroma Store
vectorstore = Chroma(embedding_function=OpenAIEmbeddings(), persist_directory="./vector_store")

## Initialize a RecursiveCharacterTextSplitter for Parent Splitter
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap = 50)

## Initialize a RecursiveCharacterTextSplitter for Child Splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap = 50)

## Initialize Parent Document Retriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=parent_doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
    
)

## Add Documents to Parent Document Retriever
parent_retriever.add_documents(docs, ids=None)


#### Invoke ParentDocumentRetriever

In [22]:
## Invoke Parent Document Retriever
parent_docs = parent_retriever.invoke("What is impact of global warming?")

## Print retrieved documents one by one
for i, doc in enumerate(parent_docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: Global warming refers to the long-term rise in Earth’s average temperature due to the buildup of greenhouse gases in the atmosphere. Human activities such as burning coal, oil, and gas, along with deforestation, release large amounts of carbon dioxide and other heat-trapping gases. This causes the planet to heat up beyond natural levels, leading to melting glaciers, rising sea levels, more frequent heatwaves, and changes in rainfall patterns. If left unchecked, global warming threatens ecosystems, human health, and the balance of life on Earth.
---
Document 2
Content: planet to heat up beyond natural levels, leading to melting glaciers, rising sea levels, more frequent heatwaves, and changes in rainfall patterns. If left unchecked, global warming threatens
---


## <font color="#3366FF">BM25 Search Retriever</font>

### Example Of BM25Retriever

In [23]:
from langchain_core.documents import Document
from langchain_classic.retrievers import BM25Retriever

## List of LangChain Documents
documents = [
    Document(
        page_content="Mars is often called the Red Planet due to its reddish appearance.",
    ),
    Document(
        page_content="The Hubble Space Telescope has provided stunning images of distant galaxies.",
    ),
    Document(
        page_content="Saturn is famous for its large and bright ring system.",
    ),
    Document(
        page_content="Regular exercise helps improve cardiovascular health and stamina.",
    ),
    Document(
        page_content="A balanced diet rich in fruits and vegetables boosts the immune system.",
    ),
    Document(
        page_content="Sleep is essential for maintaining both physical and mental well-being.",
    )
]

## Initialize BM25 Retriever
retriever = BM25Retriever.from_documents(documents, k=2)

## Invoke BM25Retriever
docs = retriever.invoke("which planet has ring")

## Print retrieved documents one by one
for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: Saturn is famous for its large and bright ring system.
---
Document 2
Content: The Hubble Space Telescope has provided stunning images of distant galaxies.
---


## <font color="#3366FF">TF-IDF Retriever</font>

### Example Of TFIDFRetriever

In [24]:
from langchain_core.documents import Document
from langchain_classic.retrievers import TFIDFRetriever

## List of LangChain Documents
documents = [
    Document(
        page_content="Mars is often called the Red Planet due to its reddish appearance.",
    ),
    Document(
        page_content="The Hubble Space Telescope has provided stunning images of distant galaxies.",
    ),
    Document(
        page_content="Saturn is famous for its large and bright ring system.",
    ),
    Document(
        page_content="Regular exercise helps improve cardiovascular health and stamina.",
    ),
    Document(
        page_content="A balanced diet rich in fruits and vegetables boosts the immune system.",
    ),
    Document(
        page_content="Sleep is essential for maintaining both physical and mental well-being.",
    )
]

## Initialize TF-IDF Retriever
retriever = TFIDFRetriever.from_documents(documents, k=2)

## Invoke TFIDFRetriever
docs = retriever.invoke("which planet has ring")

## Print retrieved documents one by one
for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: Saturn is famous for its large and bright ring system.
---
Document 2
Content: The Hubble Space Telescope has provided stunning images of distant galaxies.
---


## <font color="#3366FF">Ensemble Retriever</font>

### Example Of EnsembleRetriever

#### Setup Of EnsembleRetriever

In [25]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers import EnsembleRetriever

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## List of LangChain Documents
documents = [
    Document(
        page_content="Mars is often called the Red Planet due to its reddish appearance.",
    ),
    Document(
        page_content="The Hubble Space Telescope has provided stunning images of distant galaxies.",
    ),
    Document(
        page_content="Saturn is famous for its large and bright ring system.",
    ),
    Document(
        page_content="Regular exercise helps improve cardiovascular health and stamina.",
    ),
    Document(
        page_content="A balanced diet rich in fruits and vegetables boosts the immune system.",
    ),
    Document(
        page_content="Sleep is essential for maintaining both physical and mental well-being.",
    )
]

## Create Chroma Vector Store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=OpenAIEmbeddings()
)

## Create vector store with search type as "similarity"
retriever_similarity = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

## Create vector store with search type as "mmr"
retriever_mmr = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2})

## Create vector store with search type as "similarity_score_threshold"
retriever_similarity_score = vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.7})

# Initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(retrievers=[retriever_similarity, retriever_mmr, retriever_similarity_score])


### Invoke EnsembleRetriever

In [26]:
## Invoke EnsembleRetriever
docs = ensemble_retriever.invoke("which planet has ring")

## Print retrieved documents one by one
for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(f"Content: {doc.page_content}")
    print("---")


Document 1
Content: Saturn is famous for its large and bright ring system.
---
Document 2
Content: Top Analyst Stocks Top Smart Score Stocks Dividend CenterBest Dividend Stocks Top Insiders Stocks Top Stocks Compare stocks Stock Screener Penny Stock Screener Top Analyst Stocks AI Analyst Top Stocks Top Insiders Stocks Compare Stocks Trending Stocks Dividend Stocks Dividend Calculator Best Dividend Stocks Best High Yield Dividend Stocks Dividend Stock Comparison Dividend Calculator STOCKS The pressures on Apple’s stock price this year have come from a litany of sources, ranging from worries over tariffs that could inflate iPhone manufacturing costs to frustration with the company’s slow progress in AI. Overall we can expect increase of 1-3% 
over next 1 month after launch of Iphone 17.
---
